In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

PREPROCESSED_ROOT = Path(
    "/kaggle/input/datasets/mjayanth007/aptos-preprocessed/aptos_preprocessed"
)

TRAIN_DIR = PREPROCESSED_ROOT / "train"
TEST_DIR = PREPROCESSED_ROOT / "test"
LABELS_CSV = PREPROCESSED_ROOT / "train_labels.csv"

print("\nTraining directory exists:", TRAIN_DIR.exists())
print("Test directory exists:", TEST_DIR.exists())
print("Labels CSV exists:", LABELS_CSV.exists())

assert TRAIN_DIR.exists(), "Training directory not found."
assert LABELS_CSV.exists(), "Training labels not found."

labels_df = pd.read_csv(LABELS_CSV)

print("\nLabels shape:", labels_df.shape)
print("Columns:", labels_df.columns.tolist())
print("\nGrade distribution:")
print(labels_df["diagnosis"].value_counts().sort_index())

TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Training directory exists: True
Test directory exists: True
Labels CSV exists: True

Labels shape: (3662, 2)
Columns: ['id_code', 'diagnosis']

Grade distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split
from pathlib import Path

IMAGE_SIZE = 224
BATCH_SIZE = 32
RANDOM_SEED = 42
NUM_CLASSES = 5

labels_df["image_path"] = labels_df["id_code"].apply(
    lambda image_id: str(TRAIN_DIR / f"{image_id}.jpg")
)

# Verify image-label mapping
labels_df["file_exists"] = labels_df["image_path"].apply(
    lambda path: Path(path).is_file()
)

print("Missing images:", (~labels_df["file_exists"]).sum())

assert labels_df["file_exists"].all(), "Some labeled images are missing."
assert labels_df["diagnosis"].between(0, 4).all(), "Unexpected labels found."

data_df = labels_df.drop(columns="file_exists")

train_df, val_df = train_test_split(
    data_df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=data_df["diagnosis"]
)

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))

print("\nTraining grade distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

print("\nValidation grade distribution:")
print(val_df["diagnosis"].value_counts().sort_index())

Missing images: 0

Training images: 2929
Validation images: 733

Training grade distribution:
diagnosis
0    1444
1     296
2     799
3     154
4     236
Name: count, dtype: int64

Validation grade distribution:
diagnosis
0    361
1     74
2    200
3     39
4     59
Name: count, dtype: int64


In [3]:
import tensorflow as tf
import numpy as np

AUTOTUNE = tf.data.AUTOTUNE

def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    label = tf.cast(label, tf.int32)
    return image, label

def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["diagnosis"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Pixel range:", float(tf.reduce_min(images)),
          "to", float(tf.reduce_max(images)))

I0000 00:00:1792049542.692033      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792049542.695836      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Training batches: 92
Validation batches: 23
Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Image dtype: <dtype: 'float32'>
Pixel range: 0.0 to 255.0


In [4]:
import tensorflow as tf

NUM_CLASSES = 5

base_model = tf.keras.applications.EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

efficientnet_model = tf.keras.Model(inputs, outputs)

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

efficientnet_model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [5]:
from pathlib import Path
import tensorflow as tf

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "efficientnetb0_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10


2026-10-15 07:32:46.521314: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:32:46.664827: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:32:47.013689: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:32:47.156143: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:32:47.298764: E external/local_xla/xla/stream_

91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.5221 - loss: 1.2156

2026-10-15 07:33:08.613821: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:08.754840: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:09.086591: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:09.229389: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:10.003075: E external/local_xla/xla/stream_

92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step - accuracy: 0.5233 - loss: 1.2132

2026-10-15 07:33:29.710774: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:29.854074: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:30.202903: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:30.345875: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-15 07:33:31.124813: E external/local_xla/xla/stream_


Epoch 1: val_loss improved from None to 0.76254, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_best.keras

Epoch 1: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 67s 446ms/step - accuracy: 0.6361 - loss: 0.9938 - val_accuracy: 0.7285 - val_loss: 0.7625 - learning_rate: 0.0010
Epoch 2/10
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7328 - loss: 0.7417
Epoch 2: val_loss improved from 0.76254 to 0.69716, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_best.keras

Epoch 2: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.7402 - loss: 0.7235 - val_accuracy: 0.7449 - val_loss: 0.6972 - learning_rate: 0.0010
Epoch 3/10
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.7562 - loss: 0.6640
Epoch 3: val_loss improved from 0.69716 to 0.66798, saving model to /kaggle/working/efficien

In [6]:

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pathlib import Path

# Load the best saved model
BEST_MODEL_PATH = Path("/kaggle/working/efficientnet_outputs/efficientnetb0_best.keras")
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

# Generate predictions
y_true = np.concatenate([labels.numpy() for _, labels in val_ds])
y_prob = best_model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# Class names
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Validation accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

print("Confusion Matrix:\n")
display(cm_df)


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 302ms/step
Validation accuracy: 0.7831

Classification Report:

                  precision    recall  f1-score   support

           No DR     0.9405    0.9640    0.9521       361
            Mild     0.4853    0.4459    0.4648        74
        Moderate     0.6640    0.8300    0.7378       200
          Severe     0.6316    0.3077    0.4138        39
Proliferative DR     0.5769    0.2542    0.3529        59

        accuracy                         0.7831       733
       macro avg     0.6597    0.5604    0.5843       733
    weighted avg     0.7734    0.7831    0.7676       733

Confusion Matrix:



,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,348,7,6,0,0
Mild,14,33,26,0,1
Moderate,8,19,166,3,4
Severe,0,1,20,12,6
Proliferative DR,0,8,32,4,15


In [7]:

import json
from pathlib import Path
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

with open(OUTPUT_DIR / "efficientnet_classification_report.json", "w") as f:
    json.dump(report, f, indent=4)

cm_df.to_csv(OUTPUT_DIR / "efficientnet_confusion_matrix.csv")

summary = {
    "model": "EfficientNetB0",
    "validation_images": int(len(y_true)),
    "validation_accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(report["macro avg"]["f1-score"]),
    "weighted_f1": float(report["weighted avg"]["f1-score"]),
    "macro_recall": float(report["macro avg"]["recall"])
}

with open(OUTPUT_DIR / "efficientnet_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation results saved:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)


Evaluation results saved:
efficientnet_classification_report.json
efficientnet_confusion_matrix.csv
efficientnet_summary.json
efficientnetb0_best.keras


In [8]:

# Fine-tune the final layers of EfficientNetB0

base_model = best_model.get_layer("efficientnetb0")
base_model.trainable = True

# Keep earlier layers frozen; unfreeze the final 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

best_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Total trainable parameters:",
      sum(tf.keras.backend.count_params(w)
          for w in best_model.trainable_weights))
print("Fine-tuning setup complete.")


Total trainable parameters: 1489765
Fine-tuning setup complete.


In [9]:

from pathlib import Path
import tensorflow as tf

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
FINETUNED_MODEL_PATH = OUTPUT_DIR / "efficientnetb0_finetuned_best.keras"

finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FINETUNED_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

finetune_history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=finetune_callbacks
)

print("Fine-tuning complete.")
print("Best checkpoint:", FINETUNED_MODEL_PATH)


Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.8095 - loss: 0.5143
Epoch 1: val_loss improved from None to 0.59485, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_finetuned_best.keras

Epoch 1: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_finetuned_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 48s 292ms/step - accuracy: 0.8170 - loss: 0.5045 - val_accuracy: 0.7872 - val_loss: 0.5948 - learning_rate: 1.0000e-05
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8334 - loss: 0.4853
Epoch 2: val_loss did not improve from 0.59485
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.8211 - loss: 0.5028 - val_accuracy: 0.7790 - val_loss: 0.5949 - learning_rate: 1.0000e-05
Epoch 3/10
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8135 - loss: 0.4949
Epoch 3: val_loss improved from 0.59485 to 0.58980, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_finetuned_best.keras

Epoch 3: finished s

In [10]:

import numpy as np
import pandas as pd
import tensorflow as tf
import json
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
FINETUNED_MODEL_PATH = OUTPUT_DIR / "efficientnetb0_finetuned_best.keras"

finetuned_model = tf.keras.models.load_model(FINETUNED_MODEL_PATH)

y_true_finetuned = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

y_prob_finetuned = finetuned_model.predict(val_ds, verbose=1)
y_pred_finetuned = np.argmax(y_prob_finetuned, axis=1)

class_names = [
    "No DR", "Mild", "Moderate", "Severe", "Proliferative DR"
]

report_finetuned = classification_report(
    y_true_finetuned,
    y_pred_finetuned,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

print("Validation accuracy:",
      round(accuracy_score(y_true_finetuned, y_pred_finetuned), 4))

print("\nClassification report:\n")
print(classification_report(
    y_true_finetuned,
    y_pred_finetuned,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    digits=4,
    zero_division=0
))

cm_finetuned = confusion_matrix(
    y_true_finetuned, y_pred_finetuned, labels=[0, 1, 2, 3, 4]
)

cm_finetuned_df = pd.DataFrame(
    cm_finetuned, index=class_names, columns=class_names
)
display(cm_finetuned_df)

with open(OUTPUT_DIR / "efficientnet_finetuned_report.json", "w") as f:
    json.dump(report_finetuned, f, indent=4)

cm_finetuned_df.to_csv(
    OUTPUT_DIR / "efficientnet_finetuned_confusion_matrix.csv"
)

print("\nFine-tuned evaluation files saved.")


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 297ms/step
Validation accuracy: 0.7872

Classification report:

                  precision    recall  f1-score   support

           No DR     0.9431    0.9640    0.9534       361
            Mild     0.5143    0.4865    0.5000        74
        Moderate     0.6910    0.8050    0.7436       200
          Severe     0.5600    0.3590    0.4375        39
Proliferative DR     0.5000    0.3051    0.3789        59

        accuracy                         0.7872       733
       macro avg     0.6417    0.5839    0.6027       733
    weighted avg     0.7750    0.7872    0.7767       733



,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,348,9,4,0,0
Mild,13,36,24,0,1
Moderate,8,17,161,6,8
Severe,0,1,15,14,9
Proliferative DR,0,7,29,5,18



Fine-tuned evaluation files saved.


In [11]:

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3, 4])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].to_numpy()
)

CLASS_WEIGHT_DICT = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights)
}

print("Class weights:")
for cls, weight in CLASS_WEIGHT_DICT.items():
    print(f"Grade {cls}: {weight:.3f}")


Class weights:
Grade 0: 0.406
Grade 1: 1.979
Grade 2: 0.733
Grade 3: 3.804
Grade 4: 2.482


In [12]:

from pathlib import Path
import tensorflow as tf

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
WEIGHTED_MODEL_PATH = OUTPUT_DIR / "efficientnetb0_weighted_best.keras"

# Start from the best previously fine-tuned checkpoint
weighted_model = tf.keras.models.load_model(
    OUTPUT_DIR / "efficientnetb0_finetuned_best.keras"
)

# Keep the same fine-tuning strategy
weighted_base = weighted_model.get_layer("efficientnetb0")
weighted_base.trainable = True

for layer in weighted_base.layers[:-30]:
    layer.trainable = False

for layer in weighted_base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

weighted_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

weighted_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(WEIGHTED_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Class-weighted model prepared.")
print("Checkpoint:", WEIGHTED_MODEL_PATH)



Class-weighted model prepared.
Checkpoint: /kaggle/working/efficientnet_outputs/efficientnetb0_weighted_best.keras


In [13]:

weighted_history = weighted_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=weighted_callbacks
)

print("Class-weighted fine-tuning complete.")
print("Best checkpoint:", WEIGHTED_MODEL_PATH)


Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.8180 - loss: 0.7728
Epoch 1: val_loss improved from None to 0.65118, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_weighted_best.keras

Epoch 1: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_weighted_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 47s 288ms/step - accuracy: 0.8071 - loss: 0.7264 - val_accuracy: 0.7490 - val_loss: 0.6512 - learning_rate: 1.0000e-05
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8035 - loss: 0.7319
Epoch 2: val_loss did not improve from 0.65118
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.8020 - loss: 0.7135 - val_accuracy: 0.7422 - val_loss: 0.6535 - learning_rate: 1.0000e-05
Epoch 3/10
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8034 - loss: 0.6961
Epoch 3: val_loss did not improve from 0.65118

Epoch 3: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/ste

In [14]:

import numpy as np
import pandas as pd
import tensorflow as tf
import json
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
WEIGHTED_MODEL_PATH = OUTPUT_DIR / "efficientnetb0_weighted_best.keras"

weighted_best_model = tf.keras.models.load_model(WEIGHTED_MODEL_PATH)

y_true_weighted = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

y_prob_weighted = weighted_best_model.predict(val_ds, verbose=1)
y_pred_weighted = np.argmax(y_prob_weighted, axis=1)

class_names = [
    "No DR", "Mild", "Moderate", "Severe", "Proliferative DR"
]

print("Validation accuracy:",
      round(accuracy_score(y_true_weighted, y_pred_weighted), 4))

print("\nClassification report:\n")
print(classification_report(
    y_true_weighted,
    y_pred_weighted,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    digits=4,
    zero_division=0
))

cm_weighted = confusion_matrix(
    y_true_weighted, y_pred_weighted, labels=[0, 1, 2, 3, 4]
)

cm_weighted_df = pd.DataFrame(
    cm_weighted, index=class_names, columns=class_names
)
display(cm_weighted_df)

report_weighted = classification_report(
    y_true_weighted,
    y_pred_weighted,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

with open(OUTPUT_DIR / "efficientnet_weighted_report.json", "w") as f:
    json.dump(report_weighted, f, indent=4)

cm_weighted_df.to_csv(
    OUTPUT_DIR / "efficientnet_weighted_confusion_matrix.csv"
)

print("\nWeighted evaluation results saved.")


23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 299ms/step
Validation accuracy: 0.749

Classification report:

                  precision    recall  f1-score   support

           No DR     0.9638    0.9584    0.9611       361
            Mild     0.4386    0.6757    0.5319        74
        Moderate     0.7676    0.5450    0.6374       200
          Severe     0.3067    0.5897    0.4035        39
Proliferative DR     0.4884    0.3559    0.4118        59

        accuracy                         0.7490       733
       macro avg     0.5930    0.6250    0.5891       733
    weighted avg     0.7840    0.7490    0.7556       733



,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,346,14,1,0,0
Mild,9,50,12,1,2
Moderate,4,37,109,37,13
Severe,0,2,7,23,7
Proliferative DR,0,11,13,14,21



Weighted evaluation results saved.


In [15]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Actual screening labels:
# 0 = non-referable (original grades 0 and 1)
# 1 = referable (original grades 2, 3 and 4)
y_binary = (y_true_finetuned >= 2).astype(int)

model_probabilities = {
    "EfficientNet baseline": y_prob,
    "EfficientNet fine-tuned": y_prob_finetuned,
    "EfficientNet class-weighted": y_prob_weighted
}

results = []

for model_name, probabilities in model_probabilities.items():
    # Probability of referable DR = P(grade 2) + P(grade 3) + P(grade 4)
    referable_probability = probabilities[:, 2:].sum(axis=1)

    # Initial screening threshold; we'll tune it after comparison
    binary_predictions = (referable_probability >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_binary, binary_predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_binary, binary_predictions),
        "Referable sensitivity": recall_score(
            y_binary, binary_predictions, zero_division=0
        ),
        "Referable specificity": tn / (tn + fp) if tn + fp else 0,
        "Referable precision": precision_score(
            y_binary, binary_predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_binary, binary_predictions, zero_division=0
        ),
        "False negatives": int(fn),
        "False positives": int(fp)
    })

binary_comparison = pd.DataFrame(results)
display(binary_comparison.round(4))


,Model,Accuracy,Referable sensitivity,Referable specificity,Referable precision,F1-score,False negatives,False positives
0,EfficientNet baseline,0.9100,0.8926,0.9218,0.8867,0.8896,32,34
1,EfficientNet fine-tuned,0.9141,0.8960,0.9264,0.8930,0.8945,31,32
2,EfficientNet class-weighted,0.9195,0.8691,0.9540,0.9283,0.8977,39,20


In [16]:

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Ground truth: 0 = non-referable, 1 = referable
y_binary = (y_true >= 2).astype(int)

# Sum probabilities for grades 2, 3 and 4
referable_probability = y_prob[:, 2:].sum(axis=1)

threshold_results = []

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    predictions = (referable_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_binary, predictions, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "False negatives": int(fn),
        "False positives": int(fp),
        "Referral rate": (tp + fp) / len(y_binary)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


,Threshold,Sensitivity,Specificity,False negatives,False positives,Referral rate
0,0.20,0.9799,0.8299,6,74,0.4993
1,0.25,0.9765,0.8437,7,68,0.4898
2,0.30,0.9698,0.8690,9,57,0.4720
3,0.35,0.9597,0.8874,12,49,0.4570
4,0.40,0.9396,0.8966,18,45,0.4434
5,0.45,0.9262,0.9080,22,40,0.4311
6,0.50,0.8926,0.9218,32,34,0.4093


In [17]:

import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

grade_names = [
    "No DR", "Mild", "Moderate", "Severe", "Proliferative DR"
]

# Use the fine-tuned five-class model predictions
cm_finetuned = confusion_matrix(
    y_true_finetuned,
    y_pred_finetuned,
    labels=[0, 1, 2, 3, 4]
)

error_rows = []

for actual in range(5):
    for predicted in range(5):
        if actual != predicted:
            count = int(cm_finetuned[actual, predicted])
            if count > 0:
                error_rows.append({
                    "Actual grade": grade_names[actual],
                    "Predicted grade": grade_names[predicted],
                    "Count": count
                })

errors_df = pd.DataFrame(error_rows).sort_values(
    "Count", ascending=False
)

print("Five-class confusion matrix:")
display(pd.DataFrame(
    cm_finetuned,
    index=grade_names,
    columns=grade_names
))

print("\nMost frequent classification errors:")
display(errors_df.head(10))


Five-class confusion matrix:


,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,348,9,4,0,0
Mild,13,36,24,0,1
Moderate,8,17,161,6,8
Severe,0,1,15,14,9
Proliferative DR,0,7,29,5,18



Most frequent classification errors:


,Actual grade,Predicted grade,Count
13,Proliferative DR,Moderate,29
3,Mild,Moderate,24
6,Moderate,Mild,17
10,Severe,Moderate,15
2,Mild,No DR,13
0,No DR,Mild,9
11,Severe,Proliferative DR,9
5,Moderate,No DR,8
8,Moderate,Proliferative DR,8
12,Proliferative DR,Mild,7


### Experiment: Improve Five-Class DR Classification

**Problem identified:** The fine-tuned EfficientNetB0 frequently confuses Mild DR with Moderate DR and predicts Moderate DR for many Severe and Proliferative DR images.

**Proposed experiment:** Fine-tune the existing five-class EfficientNetB0 model using moderate data augmentation and class-weighted training.

**Objective:** Improve macro F1-score and recall for Mild, Severe, and Proliferative DR while monitoring overall accuracy and referable-DR sensitivity.

**Evaluation:** Use the same stratified training/validation split and compare the new model against the existing fine-tuned and class-weighted checkpoints. Keep the current model checkpoints unchanged.

**Screening rule:** Predict the five-class grade first. Map grades 0–1 to non-referable and grades 2–4 to referable. A referral threshold of 0.40 remains a candidate for evaluation, not a clinically validated cutoff.

In [18]:

import tensorflow as tf
import numpy as np
from pathlib import Path

# Keep existing checkpoints safe
OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
AUGMENTED_MODEL_PATH = OUTPUT_DIR / "efficientnetb0_augmented_weighted_best.keras"

# Moderate augmentation applied only during training
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
], name="fundus_augmentation")

print("Augmentation pipeline prepared.")
print("Output checkpoint:", AUGMENTED_MODEL_PATH)


Augmentation pipeline prepared.
Output checkpoint: /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_weighted_best.keras


In [19]:

# Augmentation is applied during training, not saved to disk.
# Labels remain unchanged.

def load_and_augment(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32)

    image = data_augmentation(image, training=True)

    label = tf.cast(label, tf.int32)
    return image, label


def create_augmented_dataset(dataframe):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["diagnosis"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.shuffle(
        buffer_size=len(dataframe),
        seed=RANDOM_SEED,
        reshuffle_each_iteration=True
    )
    dataset = dataset.map(
        load_and_augment,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


augmented_train_ds = create_augmented_dataset(train_df)

# Check one batch
for images, labels in augmented_train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Labels:", labels.numpy())

print("Augmented training dataset ready.")
print("Validation dataset remains unchanged:", val_ds is not None)


Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Image dtype: <dtype: 'float32'>
Labels: [0 2 0 0 3 0 4 3 4 0 2 0 4 0 0 2 4 0 2 0 0 0 0 2 0 0 2 0 0 0 2 2]
Augmented training dataset ready.
Validation dataset remains unchanged: True


In [20]:

# Load the best five-class fine-tuned checkpoint
augmented_model = tf.keras.models.load_model(
    OUTPUT_DIR / "efficientnetb0_finetuned_best.keras"
)

# Fine-tune the final layers
augmented_base = augmented_model.get_layer("efficientnetb0")
augmented_base.trainable = True

for layer in augmented_base.layers[:-30]:
    layer.trainable = False

for layer in augmented_base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

augmented_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Augmented five-class model prepared.")
print("Trainable parameters:",
      sum(tf.keras.backend.count_params(w)
          for w in augmented_model.trainable_weights))


Augmented five-class model prepared.
Trainable parameters: 1489765


In [21]:

augmented_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(AUGMENTED_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

augmented_history = augmented_model.fit(
    augmented_train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=augmented_callbacks
)

print("Augmented, class-weighted training complete.")
print("Best checkpoint:", AUGMENTED_MODEL_PATH)


Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - accuracy: 0.7717 - loss: 0.9615
Epoch 1: val_loss improved from None to 0.79455, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_weighted_best.keras

Epoch 1: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_weighted_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 60s 423ms/step - accuracy: 0.7654 - loss: 0.8760 - val_accuracy: 0.6767 - val_loss: 0.7945 - learning_rate: 1.0000e-05
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - accuracy: 0.7663 - loss: 0.7924
Epoch 2: val_loss improved from 0.79455 to 0.76468, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_weighted_best.keras

Epoch 2: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_weighted_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 207ms/step - accuracy: 0.7579 - loss: 0.8111 - val_accuracy: 0.6930 - val_loss: 0.7647 - learning_rate: 1.0000e-05
Epoch

In [22]:

import numpy as np
import pandas as pd
import tensorflow as tf
import json
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")

augmented_best_model = tf.keras.models.load_model(
    OUTPUT_DIR / "efficientnetb0_augmented_weighted_best.keras"
)

y_true_augmented = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

y_prob_augmented = augmented_best_model.predict(val_ds, verbose=1)
y_pred_augmented = np.argmax(y_prob_augmented, axis=1)

grade_names = [
    "No DR", "Mild", "Moderate", "Severe", "Proliferative DR"
]

report_augmented = classification_report(
    y_true_augmented,
    y_pred_augmented,
    labels=[0, 1, 2, 3, 4],
    target_names=grade_names,
    digits=4,
    zero_division=0
)

print("Validation accuracy:",
      round(accuracy_score(y_true_augmented, y_pred_augmented), 4))
print("\nClassification report:\n")
print(report_augmented)

cm_augmented = confusion_matrix(
    y_true_augmented, y_pred_augmented, labels=[0, 1, 2, 3, 4]
)

cm_augmented_df = pd.DataFrame(
    cm_augmented, index=grade_names, columns=grade_names
)
print("Confusion matrix:")
display(cm_augmented_df)

# Save evaluation results
report_dict = classification_report(
    y_true_augmented,
    y_pred_augmented,
    labels=[0, 1, 2, 3, 4],
    target_names=grade_names,
    output_dict=True,
    zero_division=0
)

with open(OUTPUT_DIR / "efficientnet_augmented_weighted_report.json", "w") as f:
    json.dump(report_dict, f, indent=4)

cm_augmented_df.to_csv(
    OUTPUT_DIR / "efficientnet_augmented_weighted_confusion_matrix.csv"
)

print("Evaluation results saved.")


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 312ms/step
Validation accuracy: 0.7367

Classification report:

                  precision    recall  f1-score   support

           No DR     0.9637    0.9557    0.9597       361
            Mild     0.3750    0.7703    0.5044        74
        Moderate     0.8000    0.5000    0.6154       200
          Severe     0.3279    0.5128    0.4000        39
Proliferative DR     0.4865    0.3051    0.3750        59

        accuracy                         0.7367       733
       macro avg     0.5906    0.6088    0.5709       733
    weighted avg     0.7874    0.7367    0.7429       733

Confusion matrix:


,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,345,15,1,0,0
Mild,9,57,7,0,1
Moderate,4,60,100,28,8
Severe,0,4,5,20,10
Proliferative DR,0,16,12,13,18


Evaluation results saved.


In [23]:

# Controlled experiment: augmentation without class weighting

from pathlib import Path
import tensorflow as tf

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")
AUG_ONLY_PATH = OUTPUT_DIR / "efficientnetb0_augmented_only_best.keras"

# Start from the same original fine-tuned checkpoint
aug_only_model = tf.keras.models.load_model(
    OUTPUT_DIR / "efficientnetb0_finetuned_best.keras"
)

aug_only_base = aug_only_model.get_layer("efficientnetb0")
aug_only_base.trainable = True

for layer in aug_only_base.layers[:-30]:
    layer.trainable = False

for layer in aug_only_base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

aug_only_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

aug_only_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(AUG_ONLY_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Augmentation-only experiment prepared.")
print("Class weighting will NOT be used.")
print("Checkpoint:", AUG_ONLY_PATH)


Augmentation-only experiment prepared.
Class weighting will NOT be used.
Checkpoint: /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_only_best.keras


In [24]:

aug_only_history = aug_only_model.fit(
    augmented_train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=aug_only_callbacks
)

print("Augmentation-only training complete.")
print("Best checkpoint:", AUG_ONLY_PATH)


Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step - accuracy: 0.7865 - loss: 0.5524
Epoch 1: val_loss improved from None to 0.62508, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_only_best.keras

Epoch 1: finished saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_only_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 62s 440ms/step - accuracy: 0.8016 - loss: 0.5235 - val_accuracy: 0.7790 - val_loss: 0.6251 - learning_rate: 1.0000e-05
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.8071 - loss: 0.4930
Epoch 2: val_loss did not improve from 0.62508
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 200ms/step - accuracy: 0.8020 - loss: 0.5182 - val_accuracy: 0.7722 - val_loss: 0.6320 - learning_rate: 1.0000e-05
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - accuracy: 0.8131 - loss: 0.5072
Epoch 3: val_loss improved from 0.62508 to 0.61628, saving model to /kaggle/working/efficientnet_outputs/efficientnetb0_augmented_only_best.keras



In [25]:

import numpy as np
import pandas as pd
import tensorflow as tf
import json
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

OUTPUT_DIR = Path("/kaggle/working/efficientnet_outputs")

aug_only_best_model = tf.keras.models.load_model(
    OUTPUT_DIR / "efficientnetb0_augmented_only_best.keras"
)

y_true_aug_only = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

y_prob_aug_only = aug_only_best_model.predict(val_ds, verbose=1)
y_pred_aug_only = np.argmax(y_prob_aug_only, axis=1)

grade_names = [
    "No DR", "Mild", "Moderate", "Severe", "Proliferative DR"
]

print("Validation accuracy:",
      round(accuracy_score(y_true_aug_only, y_pred_aug_only), 4))

print("\nClassification report:\n")
print(classification_report(
    y_true_aug_only,
    y_pred_aug_only,
    labels=[0, 1, 2, 3, 4],
    target_names=grade_names,
    digits=4,
    zero_division=0
))

cm_aug_only = confusion_matrix(
    y_true_aug_only, y_pred_aug_only, labels=[0, 1, 2, 3, 4]
)

cm_aug_only_df = pd.DataFrame(
    cm_aug_only, index=grade_names, columns=grade_names
)
print("Confusion matrix:")
display(cm_aug_only_df)

report_aug_only = classification_report(
    y_true_aug_only,
    y_pred_aug_only,
    labels=[0, 1, 2, 3, 4],
    target_names=grade_names,
    output_dict=True,
    zero_division=0
)

with open(OUTPUT_DIR / "efficientnet_augmented_only_report.json", "w") as f:
    json.dump(report_aug_only, f, indent=4)

cm_aug_only_df.to_csv(
    OUTPUT_DIR / "efficientnet_augmented_only_confusion_matrix.csv"
)

print("Evaluation results saved.")


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 311ms/step
Validation accuracy: 0.7817

Classification report:

                  precision    recall  f1-score   support

           No DR     0.9432    0.9668    0.9549       361
            Mild     0.4565    0.5676    0.5060        74
        Moderate     0.7027    0.7800    0.7393       200
          Severe     0.4800    0.3077    0.3750        39
Proliferative DR     0.5833    0.2373    0.3373        59

        accuracy                         0.7817       733
       macro avg     0.6332    0.5719    0.5825       733
    weighted avg     0.7749    0.7817    0.7702       733

Confusion matrix:


,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,349,11,1,0,0
Mild,12,42,19,0,1
Moderate,8,27,156,7,2
Severe,0,2,18,12,7
Proliferative DR,1,10,28,6,14


Evaluation results saved.


Final Model : efficientnetbO_finetuned with val_accuracy:79.4%